# Switch Transformer — Scaling to Trillion Parameter Models with Simple and Efficient Sparsity (2022)

_Papers · Transformers & LLMs_

**Send each token to just ONE expert: huge parameter count, near-constant compute per token.**

---

This notebook builds the Switch (MoE) layer from scratch, one piece at a time. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PyTorch

### Step 1 — Walk through the router by hand

The router scores each token over the experts with a softmax, then **top-1** routing picks the single highest-scoring expert per token (`argmax`). Two summary vectors drive load balancing: $f_i$ (Eqn 5) is the fraction of tokens routed to expert $i$, and $P_i$ (Eqn 6) is the average router probability mass on expert $i$. The auxiliary loss (Eqn 4) is $\alpha N \sum_i f_i P_i$ — small when load is spread evenly, larger when it concentrates.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

torch.manual_seed(0)
np.random.seed(0)

# --- 0. Worked example: router softmax -> top-1 pick -> aux loss, N=2, T=4. ---
logits = torch.tensor([[2., 0.], [1., 0.], [0., 1.], [3., 0.]])   # 4 tokens, 2 experts
N = 2
T = logits.shape[0]

p = F.softmax(logits, dim=1)
pick = p.argmax(dim=1)                                      # top-1 (argmax)
f = torch.stack([(pick == i).float().mean() for i in range(N)])   # Eqn 5: token fraction per expert
P = p.mean(dim=0)                                           # Eqn 6: mean router prob per expert
alpha = 1e-2
aux = alpha * N * (f * P).sum()                            # Eqn 4: load-balancing penalty

print("router probs:\n", np.round(p.numpy(), 4))
print("top-1 picks:", pick.tolist())
print("f =", np.round(f.numpy(), 4), " P =", np.round(P.numpy(), 4))
print("aux loss = alpha*N*sum(f*P) =", round(aux.item(), 6))
# router probs: [[0.8808 0.1192][0.7311 0.2689][0.2689 0.7311][0.9526 0.0474]]
# top-1 picks: [0, 0, 1, 0]
# f = [0.75 0.25]  P = [0.7083 0.2917]   aux loss = 0.012083

### Step 2 — Build the Switch (MoE) layer

Each token is routed to one expert, but every expert has a fixed **capacity** = $(T/N)\times$ capacity factor (Eqn 3). Tokens beyond an expert's capacity are *dropped* (they skip the layer via the residual). The kept tokens are run through their expert and scaled by the chosen router probability (Eqn 2). The layer also returns the auxiliary balancing loss, per-expert counts, and the drop count.

In [ ]:
# --- 1. The Switch (MoE) layer: top-1 routing + capacity + aux loss, by hand. ---
class SwitchFFN(nn.Module):
    def __init__(self, d_model, d_ff, n_experts, capacity_factor=1.25, use_aux=True, alpha=1e-2):
        super().__init__()
        self.N = n_experts
        self.cf = capacity_factor
        self.use_aux = use_aux
        self.alpha = alpha
        self.router = nn.Linear(d_model, n_experts, bias=False)          # Eqn 1: h(x)=W_r x
        self.experts = nn.ModuleList([
            nn.Sequential(nn.Linear(d_model, d_ff), nn.ReLU(), nn.Linear(d_ff, d_model))
            for _ in range(n_experts)])

    def forward(self, x):                                  # x: (tokens, d_model)
        T = x.shape[0]
        p = F.softmax(self.router(x), dim=1)               # Eqn 1: router probabilities
        pick = p.argmax(dim=1)                             # top-1 route
        gate = p.gather(1, pick[:, None]).squeeze(1)        # chosen prob (Eqn 2 scale)
        capacity = int((T / self.N) * self.cf)             # Eqn 3: per-expert capacity

        out = torch.zeros_like(x)
        counts = torch.zeros(self.N)
        dropped = 0
        for i in range(self.N):
            idx = (pick == i).nonzero(as_tuple=True)[0]
            counts[i] = idx.numel()
            keep = idx[:capacity]                          # drop overflow past capacity
            dropped += max(0, idx.numel() - capacity)
            if keep.numel():                               # Eqn 2: y = p_i(x) * E_i(x)
                out[keep] = gate[keep, None] * self.experts[i](x[keep])

        f = torch.stack([(pick == i).float().mean() for i in range(self.N)])   # Eqn 5
        P = p.mean(dim=0)                                                    # Eqn 6
        aux = self.alpha * self.N * (f * P).sum() if self.use_aux else x.new_zeros(())  # Eqn 4
        return out, aux, counts, dropped

### Step 3 — Toy data and a training loop

We make four clusters of tokens (four 'types'); each type has its own target vector, so the layer should learn to send each type to a specialist expert. The `train` function fits the Switch layer with the MSE task loss plus the auxiliary balancing loss, then reports the per-expert token share and drop count on a fresh batch.

In [ ]:
# --- 2. Toy data: 4 token "types" (4 clusters); each should learn a type-specific target. ---
d_model, d_ff, N_EXP, n_types = 16, 32, 4, 4
centers = torch.randn(n_types, d_model) * 3.0
targets = torch.randn(n_types, d_model)

def make_batch(n=64):
    xs = [centers[t] + 0.3 * torch.randn(n, d_model) for t in range(n_types)]
    y = torch.tensor(sum([[t] * n for t in range(n_types)], []))
    x = torch.cat(xs, 0)
    perm = torch.randperm(x.shape[0])
    return x[perm], y[perm]

def train(use_aux, steps=600):
    torch.manual_seed(1)
    np.random.seed(1)
    layer = SwitchFFN(d_model, d_ff, N_EXP, use_aux=use_aux)
    opt = torch.optim.Adam(layer.parameters(), lr=1e-2)
    for _ in range(steps):
        x, y = make_batch()
        out, aux, counts, dropped = layer(x)
        loss = F.mse_loss(out, targets[y]) + aux           # task + balance
        opt.zero_grad()
        loss.backward()
        opt.step()
    x, y = make_batch()
    with torch.no_grad():
        out, aux, counts, dropped = layer(x)
    return (counts / counts.sum()).numpy(), dropped

### Step 4 — Ablation: with vs without the balancing loss

Train once with the auxiliary loss and once without, everything else identical, and compare how evenly the four experts are used. We summarize balance with the coefficient of variation (CV = std / mean; 0 means perfectly even). Without the balancing loss, routing tends to **collapse** onto a couple of experts, leaving others dead and dropping the overflow.

In [ ]:
frac_aux, drop_aux = train(use_aux=True)
frac_no,  drop_no  = train(use_aux=False)

cv = lambda fr: float(np.std(fr) / np.mean(fr))              # 0 = perfectly balanced
print("\nExpert token share (4 experts), fresh batch of 256 tokens:")
print("  WITH aux   :", np.round(frac_aux, 3), " CV=%.3f" % cv(frac_aux), " dropped=", drop_aux)
print("  WITHOUT aux:", np.round(frac_no, 3),  " CV=%.3f" % cv(frac_no),  " dropped=", drop_no)
# WITH aux   : [0.25  0.25  0.191 0.309]  CV=0.166  dropped= 0
# WITHOUT aux: [0.    0.25  0.75  0.   ]   CV=1.225  dropped= 112
# (Our small run, not the paper's reported numbers. Exact values vary by seed/hardware.)

## Visualize the data & results

_Does the load-balancing auxiliary loss actually even out how many tokens each expert receives?_

### Step 1 — Re-create the layer and toy data

This visualization cell stands on its own, so it re-imports and re-defines a compact `SwitchFFN` plus the same four-cluster toy data. It is the identical layer from the reference section — top-1 routing, capacity, and the optional aux loss — just written tersely so this cell runs independently.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

torch.manual_seed(0)
np.random.seed(0)

class SwitchFFN(nn.Module):
    def __init__(self, d_model, d_ff, n, cf=1.25, use_aux=True, alpha=1e-2):
        super().__init__()
        self.N, self.cf, self.use_aux, self.alpha = n, cf, use_aux, alpha
        self.router = nn.Linear(d_model, n, bias=False)
        self.experts = nn.ModuleList([nn.Sequential(
            nn.Linear(d_model, d_ff), nn.ReLU(), nn.Linear(d_ff, d_model)) for _ in range(n)])
    def forward(self, x):
        T = x.shape[0]
        p = F.softmax(self.router(x), dim=1)
        pick = p.argmax(1)
        gate = p.gather(1, pick[:, None]).squeeze(1)
        cap = int((T / self.N) * self.cf)
        out = torch.zeros_like(x)
        counts = torch.zeros(self.N)
        dropped = 0
        for i in range(self.N):
            idx = (pick == i).nonzero(as_tuple=True)[0]
            counts[i] = idx.numel()
            keep = idx[:cap]
            dropped += max(0, idx.numel() - cap)
            if keep.numel():
                out[keep] = gate[keep, None] * self.experts[i](x[keep])
        f = torch.stack([(pick == i).float().mean() for i in range(self.N)])
        P = p.mean(0)
        aux = self.alpha * self.N * (f * P).sum() if self.use_aux else x.new_zeros(())
        return out, aux, counts, dropped

d_model, d_ff, N_EXP, n_types = 16, 32, 4, 4
centers = torch.randn(n_types, d_model) * 3.0
targets = torch.randn(n_types, d_model)

def make_batch(n=64):
    xs = [centers[t] + 0.3 * torch.randn(n, d_model) for t in range(n_types)]
    y = torch.tensor(sum([[t] * n for t in range(n_types)], []))
    x = torch.cat(xs, 0)
    perm = torch.randperm(x.shape[0])
    return x[perm], y[perm]

### Step 2 — Train both ways and compare expert balance

Run the with-aux and without-aux training again and print the per-expert token shares, the coefficient of variation, and the drop count for each. The contrast is the headline result: the balancing loss spreads tokens near $1/N$ per expert with zero drops, while turning it off collapses load onto a few experts and drops the overflow.

In [ ]:
def train(use_aux, steps=600):
    torch.manual_seed(1)
    np.random.seed(1)
    layer = SwitchFFN(d_model, d_ff, N_EXP, use_aux=use_aux)
    opt = torch.optim.Adam(layer.parameters(), lr=1e-2)
    for _ in range(steps):
        x, y = make_batch()
        out, aux, c, d = layer(x)
        loss = F.mse_loss(out, targets[y]) + aux
        opt.zero_grad()
        loss.backward()
        opt.step()
    x, y = make_batch()
    with torch.no_grad():
        out, aux, c, d = layer(x)
    return (c / c.sum()).numpy(), d

fa, da = train(True)
fn, dn = train(False)
cv = lambda fr: float(np.std(fr) / np.mean(fr))
print("WITH aux   :", [round(v, 3) for v in fa], "CV=%.3f" % cv(fa), "dropped", da)
print("WITHOUT aux:", [round(v, 3) for v in fn], "CV=%.3f" % cv(fn), "dropped", dn)
# WITH aux   : [0.25, 0.25, 0.191, 0.309] CV=0.166 dropped 0
# WITHOUT aux: [0.0, 0.25, 0.75, 0.0]     CV=1.225 dropped 112
# Our small run, not the paper's number.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The aux-loss ablation. You have a working Switch layer with 4 experts that trains WITH the
            load-balancing loss and ends up using all four experts fairly evenly. You set
            use_aux=False and retrain, everything else identical. What do you expect to happen to
            the per-expert token shares, and why?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Identify what the router is now optimizing: only the task loss, with no pressure to spread tokens. — _Nothing rewards balance, so if concentrating tokens on a strong expert lowers task loss, the router will do it._
- Predict the failure mode: a few experts capture most tokens; others receive almost none (go "dead"). With a fixed capacity factor, the overloaded experts also start DROPPING overflow tokens. — _This is routing collapse &mdash; the exact failure the aux loss exists to prevent (&sect;2.2)._
- Re-enable the aux loss and confirm usage re-balances toward $1/N$ per expert. — _The $f_i P_i$ term pushes softmax mass off overloaded experts, evening out load._

**Answer:** Without the aux loss, routing collapses: usage concentrates on a couple of experts
                 while others go dead, and the overloaded experts begin dropping tokens once they exceed
                 capacity. In our small run (CODEVIZ) the WITHOUT-aux case sends one expert ~75% of tokens and
                 leaves two others at 0% (112 tokens dropped), while WITH the aux loss all four sit near 25% with
                 zero drops. The balancing loss is what makes sparse MoE actually use its capacity.

</details>

**Problem 2.** Recompute the worked example, then change one token. Keep $N=2$, $T=4$, but change token 3's
            logits from $[0,1]$ to $[2,0]$ (so it now strongly prefers expert 0). New logits:
            $[2,0],[1,0],[2,0],[3,0]$. Compute the new picks, $f$, $P$, and aux loss. Did balance get better or
            worse?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Softmax (Eqn 1): token 3 now $\to [0.8808, 0.1192]$; the others unchanged ($[0.8808,0.1192],[0.7311,0.2689],-,[0.9526,0.0474]$). — _Only token 3's logits changed; $e^{2}/(e^{2}+1)=0.8808$._
- Picks: now ALL four tokens $\arg\max$ to expert 0, so picks $=[0,0,0,0]$ and $f=[1.0,\,0.0]$. — _Every token's largest probability is on expert 0._
- $P$ (Eqn 6): $P_0 = (0.8808+0.7311+0.8808+0.9526)/4 = 3.4453/4 = 0.8613$, $P_1 = 0.1387$. Aux (Eqn 4): $N\sum f_i P_i = 2(1.0\cdot0.8613 + 0) = 1.7227$, times $\alpha=0.01 = 0.017227$. — _All load on one expert pushes $\sum f_i P_i$ up; the loss rises._

**Answer:** Now every token routes to expert 0: $f=[1.0,0.0]$, $P=[0.8613,0.1387]$, and the aux loss
                 rises to $0.017227$ &mdash; higher than the original $0.012083$, which was itself above the
                 balanced $0.01$. Balance got worse, and the loss correctly reports it: the more skewed the
                 routing, the larger the penalty. Training would push back against exactly this state.

</details>

**Problem 3.** Your batch has $T = 256$ tokens and $N = 4$ experts, with capacity factor $1.25$. (a) What is each
            expert's capacity? (b) If one expert is chosen by 100 tokens in a step, how many of its tokens are
            dropped? (c) What happens to a dropped token?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Apply Eqn 3: capacity $= (T/N)\times\text{factor} = (256/4)\times1.25 = 64\times1.25 = 80$. — _Each expert buffers 80 tokens; the factor 1.25 adds 25% slack over the even share of 64._
- Overflow $= 100 - 80 = 20$ tokens dropped for that expert. — _Tokens past the capacity buffer cannot be processed by the expert this step._
- A dropped token skips expert computation and is passed forward unchanged through the residual connection (&sect;2.2). — _The residual path means a dropped token is not lost &mdash; it just isn't transformed by this layer._

**Answer:** (a) Capacity $= (256/4)\times1.25 = 80$ tokens per expert. (b) With 100 tokens chosen, $100-80
                 = 20$ are dropped. (c) Each dropped token skips the expert and passes through the residual
                 connection unchanged. A higher capacity factor would reduce drops at the cost of more compute and
                 memory per expert; this is the slack-vs-cost trade the factor controls.

</details>